# Tymoteusz Bultrowicz - projekt regresja

Zaczynamy od przygotowania środowiska - aktualna funkcja setup enviroment jest przystosowana \
do Google Colab. Żeby utworzyć odpowiednie środowisko lokalnie, możemy przepisać komendy:

python3 -m venv .venv \
source .venv/bin/activate \
pip install "jupyter>=1.1.1" "matplotlib>=1.6.0" "optuna>=4.8.0" "pandas>=3.0.3" "plotly>=6.7.0" "scikit-learn>=1.8.0" "statsmodels>=0.14.6" "xgboost>=3.2.0"

In [30]:
import sys
import subprocess

def setup_enviroment():
    try:
        import optuna
        print("Optuna jest już zainstalowana i gotowa do użycia!")
        return True
    except ImportError:
        print("Optuna nie została znaleziona. Rozpoczynam instalację...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])
            print("Instalacja zakończona sukcesem!")
            return True
        except Exception as e:
            print(f"Wystąpił błąd podczas instalacji: {e}")
            return False

setup_enviroment()

Optuna jest już zainstalowana i gotowa do użycia!


True

Ładujemy importy i ustalamy zmienne środowiskowe - ścieżka pliku z danymi oraz ziarno losowości.

In [31]:
import pandas as pd
import numpy as np
import optuna as opt
import plotly.express as px
import plotly.graph_objects as go
import os

from sklearn.linear_model import LinearRegression, LogisticRegression, ElasticNet, Lasso, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score, f1_score

DATA_PATH = 'validation_data.csv'
R_SEED = 42

Pobieramy dane, dzielimy na X i Y, następnie na zbiór treningowo-walidacyjny oraz testowy.

In [32]:
data = pd.read_csv(DATA_PATH, delimiter=';')
Y_binary = data['Class']
Y_discrete = data['Output']
X = data.drop(columns=['Class', 'Output'])

X_train, X_test, Y_bin_train, Y_bin_test, Y_dis_train, Y_dis_test = train_test_split(X, Y_binary, Y_discrete, test_size=0.2, random_state=R_SEED)

## Modele bazowe

Tworzymy bazowe modele, dopasowujemy je do zbioru treningowego i oceniamy na zbiorze testowym...

In [33]:
# Baseline models - creating and fitting them to the train sets
baseline_linear_mdl = LinearRegression()
baseline_logistic_mdl = LogisticRegression()
baseline_linear_mdl.fit(X_train, Y_dis_train)
baseline_logistic_mdl.fit(X_train, Y_bin_train)

# Predictions...
train_pred_linear = baseline_linear_mdl.predict(X_train)
train_pred_logistic = baseline_logistic_mdl.predict(X_train)
test_pred_linear = baseline_linear_mdl.predict(X_test)
test_pred_logistic = baseline_logistic_mdl.predict(X_test)

# Scores
print("R^2 score achieved by baseline linear model: ")
print(f"Train set: {r2_score(Y_dis_train, train_pred_linear):.4f}")
print(f"Test set: {r2_score(Y_dis_test, test_pred_linear):.4f}")

print("Scores achieved by baseline logistic model: ")
print(f"Train set: accuracy - {accuracy_score(Y_bin_train, train_pred_logistic):.4f}, f1 score - {f1_score(Y_bin_train, train_pred_logistic):.4f}")
print(f"Test set: accuracy - {accuracy_score(Y_bin_test, test_pred_logistic):.4f}, f1 score - {f1_score(Y_bin_test, test_pred_logistic):.4f}")

print("Counts of instances of different", Y_bin_train.value_counts())


R^2 score achieved by baseline linear model: 
Train set: 0.6155
Test set: 0.4464
Scores achieved by baseline logistic model: 
Train set: accuracy - 0.7388, f1 score - 0.7464
Test set: accuracy - 0.5175, f1 score - 0.5394
Counts of instances of different Class
1    815
0    785
Name: count, dtype: int64


### Podsumowanie

Widzimy, że bazowe modele zdecydowanie lepiej radzą sobie z zadaniem regresji liniowej. 

Osiągnięty wynik regresji na zbiorze testowym jest dość bliski celowi wskazówki (R^2 > 0.5), różnica między wynikami treningowym i testowym \
wskazuje jednak, że model zapewne overfittuje, więc wskazuje na miejsca, gdziemożemy dokonać poprawy, stosując regularyzację.

Wyniki modelu logistycznego są z kolei naprawdę beznadziejne - wynik 51.75% accuracy na zbiorze testowym nie \
jest nawet o procent lepszy od wyniku, jaki uzyskałby trywialny model obstawiający ciągle klasę 1. (815 / 1600 ~= 50.9%).

## Zaawansowana regresja liniowa

Wpierw obejrzyjmy jak wyglądają nasze dane, interesuje nas przede wszystkim czy dane numeryczne leżą w podobnych zakresach oraz \
ich związek z wartością końcową. Dalej będziemy próbować odsiewać szum z danych, dla uzyskania lepszych wyników regresji.

In [34]:
from statsmodels.api import OLS, add_constant
print(X_train.describe(), end='\n\n')
print(Y_dis_train.describe(), end='\n\n')

# Add constant term for intercept
X_with_const = add_constant(X_train)
ols_mdl = OLS(Y_dis_train, X_with_const)

results = ols_mdl.fit()
print(results.summary())

            Input1       Input2       Input3       Input4       Input5  \
count  1600.000000  1600.000000  1600.000000  1600.000000  1600.000000   
mean     -0.013363    -0.003294    -0.006969     0.006113     0.022932   
std       0.991482     1.019664     0.997794     0.978784     0.995371   
min      -4.040194    -3.119465    -2.978635    -3.667101    -3.466724   
25%      -0.647229    -0.700466    -0.650357    -0.653767    -0.611110   
50%       0.001729    -0.058327     0.032290     0.012005    -0.021917   
75%       0.630883     0.581323     0.654171     0.625862     0.711989   
max       3.370270     5.357014     3.674250     3.872662     3.045610   

            Input6       Input7       Input8       Input9      Input10  ...  \
count  1600.000000  1600.000000  1600.000000  1600.000000  1600.000000  ...   
mean      0.005286    -0.004080    -0.006154     0.009421    -0.026958  ...   
std       1.001415     1.017545     0.993631     1.003323     1.017921  ...   
min      -3.05913

### Podsumowanie danych

Patrząc na wartości mean, std, 25%, 50% oraz 75% wnioskujemy, że dane pochodzą z rozkładu normalnego \
N(0, 1) lub rozkładu do niego zbliżonego, nie musimy zatem przejmować się skalowaniem danych, możemy stosować \
modele bazujące na rozkładzie normalnym.

Kolejnym ważnym wnioskiem jest zauważenie bardzo dużego szumu w danych. Wiele zmiennych osiąga wyniki p wskazujące na to, \
że nie przyczyniają się w żadnym stopniu do zmiennej wyjaśnianej. Ważnym krokiem zaawansowanego modelu będzie więc feature selection.

### Zabieramy się do pracy - faza treningowa

Do testowania naszej przestrzeni hiperparametrów będziemy korzystać z narzędzia optuna. \
W następnych krokach musimy zdecydować jakie modele i parametry będziemy rozważać:

1. Feature Selection - Zdecydowałem się na korzystanie z Recursive Feature Elimination. Algorytm \
rekurencyjnie odrzuca step cech wskazywanych przez model jako najsłabiej objaśniające cel. Parametryzuję go \
hiperparametrem k oznaczającym ilość cech, które mają być pozostawione jako najważniejsze.

2. Model Selection - Zdecydowałem się na przetestowanie dwóch klasycznych regularyzowanych hiperparametrem alpha \
modeli Ridge oraz Lasso. Ponadto wprowadzam ich hybrydę ElasticNet, szukając dla niej l1_ratio. Na koniec dla \ 
nieliniowych zależności próbuję dodać XGBRegressor, który korzysta z olbrzymiej gamy hiperparametrów konstrukcji \
drzew oraz resamplingu.

3. Validation process - Dla mniejszej wariancji i bardziej stabilnych rezultatów \
testowania korzystam z 5 krotnej walidacji krzyżowej.

**Adnotacja:** Czas wykonania skryptu treningowego ~1.5min na moim urządzeniu (laptop, CPU: nowszy amd ryzen)

In [35]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
import xgboost as xgb

# mute infos from optuna.
opt.logging.set_verbosity(opt.logging.WARNING)

# Set amount of trials for optuna.
SMALL_TRIAL_NUM = 40
NUM_TRIALS = 200
N_SPLITS = 5
MIN_K_VAL = 5
MAX_K_VAL = X_train.shape[1] // 2
LOW_REGULARISATION = 1e-8
HIGH_REGULARISATION = 100.0
MIN_EST = 50
MAX_EST = 250
MIN_DEPTH = 3
MAX_DEPTH = 15

k_fold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=R_SEED)

def optimize(trial, X, y):

    # Feature selection with RFE
    K = trial.suggest_int('k', MIN_K_VAL, MAX_K_VAL)
    selector = RFE(estimator=LinearRegression(), step=0.1, n_features_to_select=K)

    # Model selection and hyperparameter tuning
    model = trial.suggest_categorical('model', ['Lasso', 'Ridge', 'ElasticNet', 'XGBoost'])
    alpha = trial.suggest_float('alpha', LOW_REGULARISATION, HIGH_REGULARISATION, log=True)
    if model == 'ElasticNet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9)
        regressor = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=R_SEED)
    elif model == 'Lasso':
        regressor = Lasso(alpha=alpha, random_state=R_SEED)
    elif model == 'Ridge':
        regressor = Ridge(alpha=alpha, random_state=R_SEED)
    elif model == 'XGBoost':
        param = {
            "verbosity": 0,
            "objective": "reg:squarederror",
            "n_estimators": trial.suggest_int("n_estimators", MIN_EST, MAX_EST),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", MIN_DEPTH, MAX_DEPTH),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "min_child_weight": trial.suggest_float("min_child_weight", 0, 5),
            "reg_alpha": trial.suggest_float("reg_alpha", LOW_REGULARISATION, HIGH_REGULARISATION, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", LOW_REGULARISATION, HIGH_REGULARISATION, log=True),
        }
        regressor = xgb.XGBRegressor(**param, random_state=R_SEED, n_jobs=-1)

    # Scoring with cross-validation
    pipeline = Pipeline([('selector', selector), ('regressor', regressor)])
    scores = cross_val_score(pipeline, X, y, cv=k_fold, scoring='r2', n_jobs=-1)
    return scores.mean()

# Creating study and optimizing it with optuna.
study = opt.create_study(study_name="Maximizing R^2 on reduced features", direction='maximize')
study.optimize(lambda trial: optimize(trial, X_train, Y_dis_train), n_trials=NUM_TRIALS, show_progress_bar=True)
print("Best trial:")
trial = study.best_trial
print(f"  R^2: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

  0%|          | 0/200 [00:00<?, ?it/s]

Best trial:
  R^2: 0.4955
  Params: 
    k: 14
    model: Ridge
    alpha: 2.3848970883679352


### Testowanie wytrenowanego modelu

**Najlepszą** wartością k dla feature_selection okazała się o dziwo niska wartość k = 14 - \
**Najlepszym** modelem z kolei okazał się Ridge z w miarę wysokim parametrem regularyzacji alpha. \
Sugeruje to zapewne występowanie **liniowej zależności** między danymi, prostej ale z dużym szumem, \
skoro mocno regularyzowany liniowy model z mała wartością k okazał się zwycięski w porównaniu z  \
zaawansowanym modelem XGRegressor.

Przechodzimy do testowania modelu:

In [36]:
# Testing our best models pipeline.

best_model_name = trial.params['model']
if best_model_name == 'ElasticNet':
    best_model = ElasticNet(alpha=trial.params['alpha'], l1_ratio=trial.params['l1_ratio'], random_state=R_SEED)
elif best_model_name == 'Lasso':
    best_model = Lasso(alpha=trial.params['alpha'], random_state=R_SEED)
elif best_model_name == 'Ridge':
    best_model = Ridge(alpha=trial.params['alpha'], random_state=R_SEED)
elif best_model_name == 'XGBoost':
    param = {
        "verbosity": 0,
        "objective": "reg:squarederror",
        "n_estimators": trial.params['n_estimators'],
        "learning_rate": trial.params['learning_rate'],
        "max_depth": trial.params['max_depth'],
        "subsample": trial.params['subsample'],
        "colsample_bytree": trial.params['colsample_bytree'],
        "gamma": trial.params['gammaSVR'],
        "min_child_weight": trial.params['min_child_weight'],
        "reg_alpha": trial.params['reg_alpha'],
        "reg_lambda": trial.params['reg_lambda'],
    }
    best_model = xgb.XGBRegressor(**param, random_state=R_SEED, n_jobs=-1)
selector = RFE(estimator=LinearRegression(), step=0.1, n_features_to_select=trial.params['k'])

pipeline = Pipeline([('selector', selector), ('regressor', best_model)])
pipeline.fit(X_train, Y_dis_train)

print(f"R^2 on train set: {r2_score(Y_dis_train, pipeline.predict(X_train)):.4f}")
print(f"R^2 on test set: {r2_score(Y_dis_test, pipeline.predict(X_test)):.4f}")

R^2 on train set: 0.5133
R^2 on test set: 0.5516


### Podsumowanie

1. Wartość testowa wyższa? - \
    Niepokojące jednak po uważnym przejrzeniu kodu pod kątem data leakage stwierdziłem, że jest to wina przypadku. Z kolei \
to, że obie wartości są wyższe od tych uzyskiwanych trenowaniem przy optunie łatwo wytłumaczyć tym, że model dopasowujemy \
teraz dopasowujemy do większej ilości danych w fazie .fit (1600 zamiast 1280).

2. Porównanie do bazowego modelu - \
    Jak widać nie dosyć, że uzyskaliśmy model osiągający lepsze wyniki R^2 score o 8 p.p., to ponadto, co bardzo ważne, \
mocno zredukowaliśmy różnicę wyników między train i test setem, co wskazywało na overfit modelu bazowego, który w tym modelu \
już nie występuje.

3. Wykresy: 

In [37]:
y_pred1 = baseline_linear_mdl.predict(X_test)
y_pred2 = pipeline.predict(X_test)

def draw_plot(y_real, y_pred, title):
    # Scatter of our predictions vs real values.
    abs_error = np.abs(y_real - y_pred)
    df_plot = pd.DataFrame({
        'Rzeczywiste': y_real,
        'Przewidziane': y_pred,
        'Blad': abs_error
    })


    fig = px.scatter(
        df_plot, 
        x='Rzeczywiste', 
        y='Przewidziane',
        custom_data=['Blad'],
        title=title,
        opacity=0.7,
        template='plotly_white',
        color_discrete_sequence=['#4169E1']
    )

    # additional info to hover
    fig.update_traces(
        hovertemplate="<b>Rzeczywista:</b> %{x:.4f}<br><b>Przewidziana:</b> %{y:.4f}"
                    "<br><b>Błąd bezwzgl.:</b> %{customdata[0]:.4f}<extra></extra>"
    )

    # diagonal
    min_val = min(y_real.min(), y_pred.min())
    max_val = max(y_real.max(), y_pred.max())
    fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
                            mode='lines', line=dict(color='#DC143C', dash='dash', width=2), 
                            name='Idealna predykcja'))

    fig.update_layout(title_font_size=16, title_x=0.5)
    fig.show()

draw_plot(Y_dis_test, y_pred1, 'Rzeczywiste vs Przewidywane Wartości - Model Bazowy')
draw_plot(Y_dis_test, y_pred2, 'Rzeczywiste vs Przewidywane Wartości - Model Ulepszony')


Z wykresów widzimy, że oba modele solidnie w całkiem dobry sposób dostosowują się do kształtu danych, nie widzimy dużych różnic \
między modelem ulepszonym a zwykłym. Obydwa niestety nie potrafią dobrze wykrywać ogonów wartości przez co przydzielają \
wartość dość monotonnie, co skutkuje przesacowywaniem niskich wartości i niedoszacowaniem wyższych wartości.

## Zaawansowana klasyfikacja

Przechodzimy do problemu klasyfikacji, przygotowując się do problemu oglądamy rozkład danych:

In [21]:
print(Y_bin_train.describe(), end='\n\n')

count    1600.000000
mean        0.509375
std         0.500068
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: Class, dtype: float64



Jak widzimy częstotliwość obu klas jest jednakowa, więc nie musimy przejmować się używaniem specjalnych \
metryk oceniania / parametrów modelu dostosowanych do nierównego rozkładu obu klas.

### Trenowanie modeli

Feature selection i validation będziemy testować w bardzo podobny sposób - z nast. różnicami \
1. Jako model do estymacji w RFE korzystamy z ciężkiego modelu Random Forest, zamiast jednego z lżejszych - daje to \
lepsze wyniki na tym zbiorze treningowym.
2. Walidować będziemy korzystając z stratifiedKFold zapewniającego dobre rozrzucenie instancji obu klas między kubełki

Model selection:
Tym razem wprowadzamy 3 bardziej zaawansowane modele z dużą przestrzenią hiperparametrów:
1. Support Vector Machine - jako że dane pochodzą z rozkładu standardowego powinien się dobrze sprawdzać.
2. Random Forest Classifier
3. XGBClassifier - działający też na zbiorze drzew, ale wykorzystujący boosting.

I oczywiście wykorzystujemy dla modeli dużą gamę hiperparametrów - szczególnie dla XG. Ponadto zostawiłem do \
porównywania prosty model zwykłej regresji logistycznej.

**Adnotacja:** Czas wykonania skryptu treningowego to ~45 min na moim urządzeniu (laptop, CPU - nowszy amd ryzen) \
**Uwaga:** W chmurze jak google colab próba testowa wykonywała się aż 10x dłużej!

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

k_fold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=R_SEED)
def optimize_logistic(trial, X, y):
    # Feature selection with RFE
    K = trial.suggest_int('k', MIN_K_VAL, MAX_K_VAL)
    selector = RFE(estimator=RandomForestClassifier(random_state=R_SEED), n_features_to_select=K, step=0.1)

    # Model selection and hyperparameter tuning
    model = trial.suggest_categorical('model', ['LogisticRegression', 'RandomForest', 'SVC', 'XGBoost'])
    if model == 'LogisticRegression':
        C = trial.suggest_float('C', LOW_REGULARISATION, HIGH_REGULARISATION, log=True)
        predictor = LogisticRegression(C=C, random_state=R_SEED)
    elif model == 'RandomForest':
        n_estimators = trial.suggest_int('n_estimators', MIN_EST, MAX_EST)
        max_depth = trial.suggest_int('max_depth', MIN_DEPTH, MAX_DEPTH)
        max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
        predictor = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, 
                                           max_features=max_features, random_state=R_SEED)
    elif model == 'SVC':
        C = trial.suggest_float('C', LOW_REGULARISATION, HIGH_REGULARISATION, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        predictor = SVC(C=C, kernel=kernel, random_state=R_SEED)
    elif model == 'XGBoost':
        param = {
            "verbosity": 0,
            "objective": "binary:logistic",
            "n_estimators": trial.suggest_int("n_estimators", MIN_EST, MAX_EST),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", MIN_DEPTH, MAX_DEPTH),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", LOW_REGULARISATION, HIGH_REGULARISATION, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", LOW_REGULARISATION, HIGH_REGULARISATION, log=True),
        }
        predictor = xgb.XGBClassifier(**param, random_state=R_SEED)

    # Scoring with cross-validation
    pipeline = Pipeline([('selector', selector), ('predictor', predictor)])
    scores = cross_val_score(pipeline, X, y, cv=k_fold, n_jobs=-1, scoring='accuracy')
    return scores.mean()

# Creating study and printing best trial results.
study_logistic = opt.create_study(study_name="Maximizing accuracy for classification", direction='maximize')
study_logistic.optimize(lambda trial: optimize_logistic(trial, X_train, Y_bin_train), 
                            n_trials=NUM_TRIALS, show_progress_bar=True)
print("Best trial:")
trial_logistic = study_logistic.best_trial
print(f"  Accuracy: {trial_logistic.value:.4f}")
print("  Params: ")
for key, value in trial_logistic.params.items():
    print(f"    {key}: {value}")

  0%|          | 0/200 [00:00<?, ?it/s]

Best trial:
  Accuracy: 0.8056
  Params: 
    k: 18
    model: XGBoost
    n_estimators: 238
    learning_rate: 0.06078986677171414
    max_depth: 12
    subsample: 0.7642813787748313
    colsample_bytree: 0.9173566693110614
    reg_alpha: 1.0709876283179954e-06
    reg_lambda: 0.004812680638434889


### Testowanie

Testujemy najlepszy uzyskany przez nas model - jak spodziewane okazał się nim jeden z bardziej \
zaawansowanych modeli XGBoost.

In [23]:
# best selector
best_k = trial_logistic.params['k']
selector = RFE(estimator=RandomForestClassifier(random_state=R_SEED), n_features_to_select=best_k, step=0.1)

# best model
best_model_name = trial_logistic.params['model']
if best_model_name == 'LogisticRegression':
    best_model = LogisticRegression(C=trial_logistic.params['C'], random_state=R_SEED)
elif best_model_name == 'RandomForest':
    best_model = RandomForestClassifier(n_estimators=trial_logistic.params['n_estimators'], 
                                        max_depth=trial_logistic.params['max_depth'], 
                                        max_features=trial_logistic.params['max_features'], random_state=R_SEED)
elif best_model_name == 'SVC':
    best_model = SVC(C=trial_logistic.params['C'], kernel=trial_logistic.params['kernel'], 
                     random_state=R_SEED, probability=True)
elif best_model_name == 'XGBoost':
    param = {
        "verbosity": 0,
        "objective": "binary:logistic",
        "n_estimators": trial_logistic.params['n_estimators'],
        "learning_rate": trial_logistic.params['learning_rate'],
        "max_depth": trial_logistic.params['max_depth'],
        "subsample": trial_logistic.params['subsample'],
        "colsample_bytree": trial_logistic.params['colsample_bytree'],
        "reg_alpha": trial_logistic.params['reg_alpha'],
        "reg_lambda": trial_logistic.params['reg_lambda'],
    }
    best_model = xgb.XGBClassifier(**param, random_state=R_SEED)
pipeline = Pipeline([('selector', selector), ('predictor', best_model)])

# scoring
pipeline.fit(X_train, Y_bin_train)
train_acc = accuracy_score(Y_bin_train, pipeline.predict(X_train))
test_acc = accuracy_score(Y_bin_test, pipeline.predict(X_test))
print(f"Best model train accuracy: {train_acc:.4f}")
print(f"Best model test accuracy: {test_acc:.4f}")

Best model train accuracy: 1.0000
Best model test accuracy: 0.8475


### Bonus - Voting Classifier
Choć wyniki uzyskane przez XGBoost są dla nas satysfakcjonujące (> 0.8 na test secie), \
to próbujemy jeszcze dodatkowo ulepszyć nasz model.

W tym celu tworzymy soft voting classifier łączący go z modelem działającym w kompletnie inny sposób - SVMem:

**Adnotacja:** Czas wykonania ~8min.

In [24]:
from sklearn.ensemble import VotingClassifier 

# Szukamy optymalnego SVC analogicznie do tego jak szukaliśmy optymalnego modelu klasyfikacyjnego.
def opt_SVC(trial, X, y):
    C = trial.suggest_float('C', LOW_REGULARISATION, HIGH_REGULARISATION, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    predictor = SVC(C=C, kernel=kernel, random_state=R_SEED)
    pipeline = Pipeline([('selector', selector), ('predictor', predictor)])
    scores = cross_val_score(pipeline, X, y, cv=k_fold, scoring='accuracy', n_jobs=-1)
    return scores.mean()

print("Optimizing SVC matching our XGBoost model...")
study_SVC = opt.create_study(study_name="Optimizing SVC", direction='maximize')
study_SVC.optimize(lambda trial: opt_SVC(trial, X_train, Y_bin_train), n_trials=SMALL_TRIAL_NUM, show_progress_bar=True)
print("Best SVC trial:")
trial_SVC = study_SVC.best_trial
print(f"  Accuracy: {trial_SVC.value:.4f}")
print("  Params: ")
for key, value in trial_SVC.params.items():
    print(f"    {key}: {value}")

svc_mdl = SVC(C=trial_SVC.params['C'], kernel=trial_SVC.params['kernel'], random_state=R_SEED, probability=True)
XGB_mdl = best_model

Optimizing SVC matching our XGBoost model...


  0%|          | 0/40 [00:00<?, ?it/s]

Best SVC trial:
  Accuracy: 0.7906
  Params: 
    C: 68.20753326709921
    kernel: rbf


### Sprawdzamy wyniki walidacyjne naszego soft votera:

In [25]:
print(f"Testing created soft-voting-classifier")
vote_mdl = VotingClassifier(estimators=[('svc', svc_mdl), ('xgb', XGB_mdl)], voting='soft')
vote_final = Pipeline([('selector', selector), ('predictor', vote_mdl)])

scores = cross_val_score(vote_final, X_train, Y_bin_train, cv=k_fold, scoring='accuracy', n_jobs=-1)
print(f"Soft Voting Classifier Accuracy: {scores.mean():.4f}")

Testing created soft-voting-classifier
Soft Voting Classifier Accuracy: 0.8100


### Finalny model

Ponieważ na zbiorze walidacyjnym model miękkiego głosowania zachował się lepiej od XGB, testujemy
go jako nasz ostateczny ulepszony model:

In [26]:
vote_final.fit(X_train, Y_bin_train)
train_acc = accuracy_score(Y_bin_train, vote_final.predict(X_train))
test_acc = accuracy_score(Y_bin_test, vote_final.predict(X_test))
print(f"Best final model train accuracy: {train_acc:.4f}")
print(f"Best final model test accuracy: {test_acc:.4f}")

Best final model train accuracy: 0.9969
Best final model test accuracy: 0.8550


### Podsumowanie

Dzięki łączonemu modelowi udało nam się poprawić wynik o blisko procent. Nasz wyspecjalizowany model \
jest w stanie przejść próg "zgadywania", z niską, ok. 50% skutecznością i uzyskiwać wynik o ponad 30% wyższy. \
Znaczy to, że jest bezwątpienia lepszy od klasycznej regresji logistycznej uzyskującej wyniki podobne do modelu, \
który stale obstawia wynik "0".


Spójrzmy jak układają się rozkład False Positives do Negatives w przewidywaniach modelu:


In [29]:
def draw_clf_plot(y_real, y_pred, title):
    y_real = np.array(y_real)
    y_pred = np.array(y_pred)
    hits_0 = np.sum((y_real == 0) & (y_pred == 0))  # True Positive
    miss_0 = np.sum((y_real == 0) & (y_pred == 1))  # False Negative
    hits_1 = np.sum((y_real == 1) & (y_pred == 1))  # True Negative
    miss_1 = np.sum((y_real == 1) & (y_pred == 0))  # False Positive
    df_grouped = pd.DataFrame({
        'Klasa': ['Class 0 (+)', 'Class 0 (+)', 'Class 1 (-)', 'Class 1 (-)'],
        'Rezultat': ['Correct', 'Missed', 'Correct', 'Missed'],
        'Ilość pred.': [hits_0, miss_0, hits_1, miss_1]
    })

    # Wykres
    fig = px.bar(
        df_grouped, 
        x='Klasa', 
        y='Ilość pred.',
        color='Rezultat',
        barmode='group',
        color_discrete_map={
            'Correct': '#2ECC71',
            'Missed': '#E74C3C'
        },
        title=title,
        template='plotly_white'
    )
    
    # Dostosowanie wyglądu
    fig.update_layout(
        title_font_size=16, 
        title_x=0.5, 
        yaxis_title='Ilość pred.',
        xaxis_title='Klasa rzeczywista',
        legend_title_text='Wynik klasyfikacji'
    )
    
    fig.show()

draw_clf_plot(Y_bin_test, baseline_logistic_mdl.predict(X_test), "Rozkład klasyfikacji - Model Bazowy")
draw_clf_plot(Y_bin_test, vote_final.predict(X_test), "Rozkład klasyfikacji - Soft Voting Classifier")

### Analiza wykresów

Poprawa jakości jest zauważalna na peirwszy rzut oka.

Ciekawe jest, że niezależnie od modelu obiekty klasy 1 wydają się prostsze do rozpoznania, \
a to obiekty klasy 0 są częściej mylone i uznawane za obiekty klasy 1.